# 01b Process ScenarioMIP data

In [11]:
%load_ext autoreload
%autoreload 2
    
from dask.distributed import Client
import warnings
warnings.filterwarnings('ignore')

from cdrmip_extremes import configs, pipeline
from cdrmip_extremes.extremes_pipeline import ExtremesPipeline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)

In [3]:
config = pipeline.PipelineConfig(
    experiments=[
        pipeline.ExperimentConfig(
            experiment_id='ssp534-over', table_id='Amon', variable_id='tas',
            anomaly_reference='period',grid_label=None,
            gwl_match_period=(2180, 2200),
            check_end_year=True, threshold=2101,
            rolling_window=21
        ),
    ],
    branch_date_overrides= {
        ('ACCESS-ESM1-5','ssp534-over'):'2041-01-01',   
    }
)

In [4]:
config = pipeline.CDRMIPPipeline(config)

In [5]:
pipe = config.run()

=== Loading ssp534-over ===
Searching for files matching: {'experiment_id': 'ssp534-over', 'table_id': 'Amon', 'variable_id': 'tas', 'grid_label': None, 'member_id': None}
Loading ACCESS-CM2 / r1i1p1f1
  2041-01-16 12:00:00 -> 2100-12-16 12:00:00
  Skipping — end year 2100 not beyond 2101
Loading ACCESS-ESM1-5 / r10i1p1f2
  2041-01-16 12:00:00 -> 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 / r11i1p1f2
  2041-01-16 12:00:00 -> 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 / r12i1p1f2
  2041-01-16 12:00:00 -> 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 / r13i1p1f2
  2041-01-16 12:00:00 -> 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 / r14i1p1f2
  2041-01-16 12:00:00 -> 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 / r15i1p1f2
  2041-01-16 12:00:00 -> 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 / r16i1p1f2
  2041-01-16 12:00:00 -> 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 / r17i1p1f2
  2041-01-16 12:00:00 -> 2300-12-16 12:00:00
Loading ACCESS-ESM1-5 / r18i1p1f2
  2041-01-16 12:00:00 -> 2300-12-16 12:00:00

In [6]:
pipe.load_gsat().compute_gsat_rolling().compute_peak_warming()

In [7]:
pipe.compute_crossing_years().compute_gwl_matching().extract_gwl_periods()

In [8]:
pipe.compute_ensemble_summaries()

  Skipping crossing_years: empty (not yet computed)
  Skipping gwl_reversibility: empty (not yet computed)
  Skipping multi-model gwl_reversibility: gwl_reversibility_median empty/not computed


## Save

In [9]:
pipe.save(
    configs.PROC_DIR,
    'ScenarioMIP',
    array_outputs={
        'gsat': (pipe.gsat_median, 'gsat'),
        'gsat_rolling': (pipe.gsat_rolling_median, 'gsat_rolling'),
        'gwl_matching_years': (pipe.gwl_matching, 'gwl_matching_years'),
        'gwl_differences': (pipe.gwl_differences_median_multimodel, 'gwl_differences'),
    },
    scalar_outputs={
    },
    overwrite=False
)

## Extreme Analysis

In [12]:
ext = ExtremesPipeline(
    regridded_picontrol=pipe.regridded_picontrol,
    regridded_concat=pipe.regridded_concat,
    gwl_matching=pipe.gwl_matching,
    gwl_window=21
)

In [13]:
ext.compute_extreme_months().compute_thresholds().extract_gwl_period_extremes()
ext.compute_exceedances(source_attr='gwl_period_extremes_matching')

  No crossing_years provided — skipping gwl_period_extremes


In [14]:
ext.load_exceedances('exceedances_matching')

In [15]:
ext.compute_branch_differences(source_attr='exceedances_matching')

In [16]:
ext.compute_multi_model_stat('exceedances_matching')
ext.compute_multi_model_stat('branch_differences_matching')

In [17]:
ext.save(
    configs.PROC_DIR,
    'ScenarioMIP',
    outputs=[
        'agreement_matching',
        'exceedances_matching_median_multimodel',
        'branch_differences_matching_median_multimodel'
    ],
    overwrite=False
)